In [183]:
import pandas as pd
import numpy as np
import openpyxl

In [184]:
df = pd.read_excel("../Datos/Originales/Prestamos_Data_Alumnos_v3.xlsx")

df = df.dropna(subset=["Prima"]).copy()

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,S97R7X,18,16000,5000,397,19,1,8.06,48,0.10,Escolar,Autónomo,Soltero,1,0,Automóvil,0,0,50.12
1,T3ZE0N,69,72673,32340,784,320,2,15.04,48,0.12,Doctorado,Tiempo parcial,Casado,0,1,Educación,0,0,187.15
2,RLGTBY,50,62116,37278,486,217,3,21.96,12,0.55,Grado Universitario,Tiempo parcial,Casado,1,0,Automóvil,1,1,800.00
3,BZ86CV,64,59846,19784,308,340,1,24.26,12,0.35,Grado Universitario,Desempleado,Divorciado,1,0,Negocios,1,0,275.82
4,5OD75M,62,28413,13751,412,476,2,5.73,36,0.14,Escolar,Autónomo,Casado,0,0,Negocios,0,0,100.07


In [185]:
df = df[df["Proposito"] == "Vivienda"]

In [186]:
df["Prima"].value_counts(normalize=True).head(10)


Prima
800.00    0.786998
711.60    0.000078
785.05    0.000078
442.98    0.000078
574.37    0.000078
681.66    0.000078
553.23    0.000078
563.94    0.000058
750.25    0.000058
493.06    0.000058
Name: proportion, dtype: float64

In [187]:
df["Prima"].describe()


count    51286.000000
mean       753.720705
std        112.326178
min        143.320000
25%        800.000000
50%        800.000000
75%        800.000000
max        800.000000
Name: Prima, dtype: float64

In [188]:
df = df[df["Prima"] != 800].copy()


In [189]:
df.duplicated(subset="ID").sum()

#duplicados

np.int64(0)

In [190]:
df.isna().sum()

#nulos

ID                      0
Edad                    0
Ingresos                0
Monto_Inicial           0
Scoring_Crediticio      0
Meses_Empleo            0
Num_Creditos            0
Ratio_Interes           0
Duracion                0
Ratio_Deuda_Ingresos    0
Estudios                0
Tipo_Jornada_Laboral    0
Estado_Civil            0
Posesion_Hipoteca       0
Personas_Cargo          0
Proposito               0
Fiador                  0
Impago                  0
Prima                   0
dtype: int64

In [191]:
df["Meses_Maximos"] = (df["Edad"] - 16) * 12
df_invalidos = df[df["Meses_Empleo"] > df["Meses_Maximos"]]

In [192]:
df = df[df["Meses_Empleo"] <= df["Meses_Maximos"]]
df = df.drop(columns="Meses_Maximos")

## Se eliminaron registros con cosas no lógicas en la variable Meses_Empleo, ya que implicaban experiencia laboral previa 
# a la edad legal mínima (16 años).

In [193]:
df = df[(df["Edad"] >= 16) & (df["Edad"] <= 100)]

# edades, no menor a 16 y mayores a 100

In [194]:
df["Tipo_Jornada_Laboral"].value_counts()

Tipo_Jornada_Laboral
Autónomo            2785
Tiempo parcial      2730
Desempleado         2706
Jornada completa    2703
Name: count, dtype: int64

In [195]:
df["Tipo_Jornada_Laboral"] = (
    df["Tipo_Jornada_Laboral"]
    .str.strip()
    .str.lower()
)

df["Tipo_Jornada_Laboral"] = df["Tipo_Jornada_Laboral"].replace({
    "autonomo": "autónomo"
})

df["Tipo_Jornada_Laboral"].value_counts()

## por si hay alguna palabra mal, en plan espacios masculino/femenino

Tipo_Jornada_Laboral
autónomo            2785
tiempo parcial      2730
desempleado         2706
jornada completa    2703
Name: count, dtype: int64

In [196]:
df[~df["Fiador"].isin([0,1])]

## tiene que ser binario

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima


In [197]:
df = df[df["Ingresos"] > 0]

## ingresos siempre positivos, no se pueden negativos

In [198]:
df = df[df["Monto_Inicial"] > 0]

## monto inicial imposible negativo

In [199]:
df = df[(df["Ratio_Deuda_Ingresos"] >= 0) & (df["Ratio_Deuda_Ingresos"] <= 1)]

# no puedes menos del 0 y no puedes deber mas de 1

In [200]:
df = df[(df["Ratio_Interes"] > 0) & (df["Ratio_Interes"] <= 100)]

# no se puede menos de 0 ni mas de 100

In [201]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10924 entries, 19 to 255345
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    10924 non-null  object 
 1   Edad                  10924 non-null  int64  
 2   Ingresos              10924 non-null  int64  
 3   Monto_Inicial         10924 non-null  int64  
 4   Scoring_Crediticio    10924 non-null  int64  
 5   Meses_Empleo          10924 non-null  int64  
 6   Num_Creditos          10924 non-null  int64  
 7   Ratio_Interes         10924 non-null  float64
 8   Duracion              10924 non-null  int64  
 9   Ratio_Deuda_Ingresos  10924 non-null  float64
 10  Estudios              10924 non-null  object 
 11  Tipo_Jornada_Laboral  10924 non-null  object 
 12  Estado_Civil          10924 non-null  object 
 13  Posesion_Hipoteca     10924 non-null  int64  
 14  Personas_Cargo        10924 non-null  int64  
 15  Proposito             

In [202]:
condiciones_invalidas = (
    ((df["Edad"] < 19) & (df["Estudios"] == "grado")) |
    ((df["Edad"] < 20) & (df["Estudios"] == "máster")) |
    ((df["Edad"] < 25) & (df["Estudios"] == "doctorado"))
)

df_estudios_invalidos = df[condiciones_invalidas]
len(df_estudios_invalidos)

0

In [203]:
df.shape

(10924, 19)

In [204]:
df.to_excel(
    "../Datos/Limpios/información_préstamos_limpio.xlsx",
    index=False
)
